# StudyReader — Smart Academic Subject Classification
## Machine Learning Internship Project Experiment Notebook

This notebook demonstrates the end-to-end Machine Learning pipeline for automatically classifying academic study materials (textbooks, lecture notes, question papers) across 15 distinct subjects spanning School (Grades 9-12) and College curricula.

### Pipeline Overview:
1. **Dataset Loading & Exploration**
2. **Text Normalization & Preprocessing**
3. **TF-IDF Feature Extraction (Unigrams + Bigrams)**
4. **Model 1: Multinomial Naive Bayes**
5. **Model 2: Logistic Regression (Softmax Regression)**
6. **Comparative Model Evaluation & Confusion Matrix**
7. **Zero-Dependency Model Export (`model-bundle.json`)**

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Add src directory to path
sys.path.append(os.path.abspath('../src'))
from preprocess import clean_text

print("Environment initialized successfully.")

### 1. Load Academic Dataset

In [ ]:
dataset_path = '../dataset/academic_subjects_dataset.csv'
df = pd.read_csv(dataset_path)
print(f"Total samples: {len(df)}")
print(f"Distinct subjects: {df['subject'].nunique()}")
df['subject'].value_counts()

### 2. Preprocess and Clean Text

In [ ]:
df['clean_text'] = df['text'].apply(clean_text)
print("Sample original text:")
print(df['text'].iloc[0][:150])
print("\nSample cleaned text:")
print(df['clean_text'].iloc[0][:150])

### 3. Stratified Train / Test Split and TF-IDF Vectorization

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df['clean_text']
y = df['subject']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=3500, sublinear_tf=True, min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} features")
print(f"Training matrix shape: {X_train_vec.shape}")
print(f"Testing matrix shape: {X_test_vec.shape}")

### 4. Train Multinomial Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_vec, y_train)
nb_preds = nb_model.predict(X_test_vec)

print(f"Multinomial Naive Bayes Accuracy: {accuracy_score(y_test, nb_preds) * 100:.2f}%")

### 5. Train Logistic Regression (Softmax)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_vec, y_train)
lr_preds = lr_model.predict(X_test_vec)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, lr_preds) * 100:.2f}%")
print("\nDetailed Classification Report:")
print(classification_report(y_test, lr_preds))

### 6. Interactive Prediction on Unseen Texts

In [ ]:
from predict import SubjectPredictor

predictor = SubjectPredictor('../models/model-bundle.json')
samples = [
    "Binary search tree with AVL balancing rotations maintains O(log n) time complexity.",
    "Database normalization into 3NF and BCNF avoids anomalies in SQL transactions.",
    "The Constitution of India guarantees Fundamental Rights and Parliamentary democracy.",
    "During photosynthesis, chloroplasts convert solar energy into chemical ATP and NADPH."
]

for s in samples:
    res = predictor.predict(s)
    print(f"Excerpt: '{s}'")
    print(f"Predicted Subject: {res['predicted_subject']} (Confidence: {res['confidence']*100:.2f}%, Status: {res['status']})\n")